# Validation of `LinearSmallStrainFEMForceField`

This notebook validates SOFA's `LinearSmallStrainFEMForceField` against FEniCS, as presented in [IntroductionValidation.ipynb](IntroductionValidation.ipynb), on the cases in `validation/cases/LinearSmallStrainFEMForceField/`. Where a case has an analytical solution, SOFA's solution is compared with it as well.

## Running the Cases

The cells below load the suite's runner and prepare the figures. Run them once before any case: every case cell below then runs on its own, in any order.

In [ ]:
# Make the plugin's packages importable from validation/docs
import sys
sys.path.insert(0, "../..")

# Hide the runner's printed report; figures still show
import os
sys.stdout = open(os.devnull, "w")

# Load the suite's runner
from validation.run import run

# For showing figures inline
%matplotlib inline

`side_by_side` shows the figures of a case's elements in one row.

In [ ]:
import base64
import io
from IPython.display import HTML, display

def side_by_side(*figures):
    images = []
    for figure in figures:
        buffer = io.BytesIO()
        figure.savefig(buffer, format="png")
        encoded = base64.b64encode(buffer.getvalue()).decode()
        images.append(f'<img src="data:image/png;base64,{encoded}" style="width: {100 / len(figures)}%">')
    display(HTML(f'<div style="display: flex">{"".join(images)}</div>'))

## Edge Elements on a 1D Bar

The bar starts at the origin and has length $L$ along $x$. The displacement $u$ is a scalar.

### Distributed Axial Load

The bar carries a uniform axial load $q$ per unit length. With Young's modulus $E$, the displacement satisfies $-E\,u'' = q$, and its exact solution is

$$u(x) = \frac{q}{E}\left(L x - \frac{x^2}{2}\right)$$

- `left` ($x = 0$): $u = 0$
- `right` ($x = L$): $t = 0$, traction-free, with nothing applied

In [ ]:
# Case: 1D bar under a distributed axial load
study = run("../cases/LinearSmallStrainFEMForceField/1D/DistributedAxialLoad")
# SOFA, FEniCS and the exact solution along the bar
study.plots["edge"]

## Triangle and Quadrilateral Elements on a 2D Beam

The beam starts at the origin, with length $L$ along $x$ and width $W$ along $y$. Each case is run on quadrilateral (`quad`) and triangle (`tri`) elements.

### Axial Compression

The right side carries a uniform compressive traction $p$. The left side is held only along $x$, so that the beam can contract freely across its width, and the stress is uniform, $\sigma_{xx} = -p$. The exact solution is

$$u_x = -\frac{p}{E}\,x, \qquad u_y = \frac{\nu p}{E}\,y$$

- `left` ($x = 0$): $u_x = 0$
- corner $(0, 0)$: $u_y = 0$
- `right` ($x = L$): $\mathbf{t} = (-p, 0)$
- `bottom` ($y = 0$) and `top` ($y = W$): $\mathbf{t} = 0$

In [ ]:
# Case: 2D beam under axial compression
study = run("../cases/LinearSmallStrainFEMForceField/2D/AxialCompression")
# quad on the left, tri on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["quad"], study.plots["tri"])

### Distributed Transverse Traction

The beam is clamped on its left side, and its top side carries a uniform downward traction $q$. This case has no exact solution, so SOFA is compared with FEniCS only.

- `left` ($x = 0$): $\mathbf{u} = 0$
- `top` ($y = W$): $\mathbf{t} = (0, -q)$
- `right` ($x = L$) and `bottom` ($y = 0$): $\mathbf{t} = 0$

In [ ]:
# Case: 2D cantilever under a distributed transverse traction
study = run("../cases/LinearSmallStrainFEMForceField/2D/DistributedTransverseTraction")
# quad on the left, tri on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["quad"], study.plots["tri"])

### Three-Point Bending

The beam rests on two rollers at $(0, 0)$ and $(L, 0)$, and a point force $P$ pushes down on the middle of its top side. The bottom of the midspan is held along $x$, which stops the beam from sliding without breaking its symmetry. This case has no exact solution, so SOFA is compared with FEniCS only.

- rollers $(0, 0)$ and $(L, 0)$: $u_y = 0$
- node $(L/2, 0)$: $u_x = 0$
- node $(L/2, W)$: force $(0, -P)$
- all sides: $\mathbf{t} = 0$

In [ ]:
# Case: 2D beam in three-point bending
study = run("../cases/LinearSmallStrainFEMForceField/2D/ThreePointBending")
# quad on the left, tri on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["quad"], study.plots["tri"])

## Tetrahedron and Hexahedron Elements on a 3D Beam

The beam starts at the origin, with length $L$ along $x$, width $W$ along $y$ and height $H$ along $z$. Each case is run on hexahedron (`hexa`) and tetrahedron (`tet`) elements.

### Axial Compression

The right face carries a uniform compressive traction $p$. The left face is held only along $x$, so that the beam can contract freely across its section, and the stress is uniform, $\sigma_{xx} = -p$. The exact solution is

$$u_x = -\frac{p}{E}\,x, \qquad u_y = \frac{\nu p}{E}\,y, \qquad u_z = \frac{\nu p}{E}\,z$$

- `left` ($x = 0$): $u_x = 0$
- corner $(0, 0, 0)$: $u_y = u_z = 0$
- corner $(0, W, 0)$: $u_z = 0$
- `right` ($x = L$): $\mathbf{t} = (-p, 0, 0)$
- the other faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D beam under axial compression
study = run("../cases/LinearSmallStrainFEMForceField/3D/AxialCompression")
# hexa on the left, tet on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["hexa"], study.plots["tet"])

### Distributed Transverse Traction

The beam is clamped on its left face, and its top face carries a uniform downward traction $q$. This case has no exact solution, so SOFA is compared with FEniCS only.

- `left` ($x = 0$): $\mathbf{u} = 0$
- `top` ($y = W$): $\mathbf{t} = (0, -q, 0)$
- the other faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D cantilever under a distributed transverse traction
study = run("../cases/LinearSmallStrainFEMForceField/3D/DistributedTransverseTraction")
# hexa on the left, tet on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["hexa"], study.plots["tet"])

### Three-Point Bending

The beam rests on a pinned line at $x = 0$ and on a roller line at $x = L$, both on its bottom face and along $z$. A force $P$ pushes down on each node of the line across the middle of its top face. This case has no exact solution, so SOFA is compared with FEniCS only.

- line $(0, 0, z)$: $u_x = u_y = 0$
- line $(L, 0, z)$: $u_y = 0$
- node $(0, 0, 0)$: $u_z = 0$
- line $(L/2, W, z)$: force $(0, -P, 0)$ on each node
- all faces: $\mathbf{t} = 0$

In [ ]:
# Case: 3D beam in three-point bending
study = run("../cases/LinearSmallStrainFEMForceField/3D/ThreePointBending")
# hexa on the left, tet on the right: a row each for SOFA, FEniCS and the exact solution if any, a column per component of u
side_by_side(study.plots["hexa"], study.plots["tet"])

## Tetrahedron Elements on a 3D Circular Beam

The beam is a cylinder of radius $R$ and length $L$ along $x$, its left face centred at the origin. `mesh.py` in each case folder meshes it with tetrahedra (`tet`) using [gmsh](https://gmsh.info), and SOFA and FEniCS both read that mesh file. The left face is clamped, and the lateral surface is traction-free. These cases have no exact solution, so SOFA is compared with FEniCS only.

### Axial Compression

The right face carries a uniform compressive traction $p$.

- left face ($x = 0$): $\mathbf{u} = 0$
- right face ($x = L$): $\mathbf{t} = (-p, 0, 0)$

In [ ]:
# Case: 3D circular beam under axial compression
study = run("../cases/LinearSmallStrainFEMForceField/3D/CircularAxialCompression")
# a row each for SOFA and FEniCS, a column per component of u
study.plots["tet"]

### Transverse Traction

The right face carries a uniform downward traction $q$, which bends the beam as a cantilever.

- left face ($x = 0$): $\mathbf{u} = 0$
- right face ($x = L$): $\mathbf{t} = (0, -q, 0)$

In [ ]:
# Case: 3D circular cantilever under a transverse traction
study = run("../cases/LinearSmallStrainFEMForceField/3D/CircularTransverseTraction")
# a row each for SOFA and FEniCS, a column per component of u
study.plots["tet"]

### Torsion

The right face carries a torque $T$ about the axis, as the traction of a circular shaft in torsion: tangential, with magnitude $T r / J$ at the distance $r$ from the axis, where $J = \pi R^4 / 2$ is the polar moment of the section.

- left face ($x = 0$): $\mathbf{u} = 0$
- right face ($x = L$): $\mathbf{t} = \frac{T}{J}\,(0, -z, y)$

In [ ]:
# Case: 3D circular beam in torsion
study = run("../cases/LinearSmallStrainFEMForceField/3D/CircularTorsion")
# a row each for SOFA and FEniCS, a column per component of u
study.plots["tet"]